# first project

**signal b**\
nested cross validation: train con cross validation su 18, poi 4 per test interno\
**signal a**\
test esterno su dati che il modello non ha mai visto

### machine learning supervisionato

- ogni segnale è un file .mat, in cui sono contenute due matrici 22x16288: g_0 e g_1
- Per la fase di training del modello usiamo il segnale B
- g_0 e g_1 rappresentano le due classi: negativi e positivi. 

- **ogni riga (2 x 22) è un pattern (campione/osservazione), ogni colonna (16288) è una _feature_ (caratteristica).**

### recupero la matrice da file .mat

In [1]:
import numpy as np

g_0_b = np.loadtxt('data/g_0_signal_b.txt')
g_1_b = np.loadtxt('data/g_1_signal_b.txt')

g_0_a = np.loadtxt('data/g_0_signal_a.txt')
g_1_a = np.loadtxt('data/g_1_signal_a.txt')

### controllo shape

In [2]:
print("shape g_0")
print(g_0_b.shape)
print("shape g_1")
print(g_1_b.shape)

shape g_0
(22, 16288)
shape g_1
(22, 16288)


### controllo spettro

In [3]:
import matplotlib.pyplot as plt

"""
Se vuoi stampare tutto
x = range(signal_a[20].shape[0])
y = signal_a[20]
"""

for i in range(32):
    plt.figure(figsize=(5,4), dpi=100)
    plt.ioff() 
    da = i*509
    a = i*509+509
    y0 = g_0_b[20][da:a]
    y1 = g_1_b[20][da:a]
    x = range(g_0_b[20][da:a].shape[0])
    
    plt.plot(x,y0)
    plt.plot(x,y1)
    #plt.savefig(f'outputs/g_0_E_g_1_signal_b/g_0_E_g_1_signal_b_{i}.png')
    plt.close()


### preparazione ingredienti

Dobbiamo addestrare un modello svm (Support Vector Machine). Dobbiamo fornirgli:
- i dati (le 16288 colonne)
- le risposte corrette (labels)

per questo concateniamo g_0 e g_1, ottenendo una matrice 44x16288; quindi formiamo un vettore delle risposte da 44 entrate: 22 zero e 22 uno. ciascuna riga del matricione è in corrispondenza con una riga del vettore labels.

In [4]:
# sono entrambe 22, ma così è più generale
N0 = g_0_b.shape[0]
N1 = g_1_b.shape[0]

In [5]:
# concateno g_0 e g_1
matricione = np.vstack((g_0_b, g_1_b))
matricione_testing = np.vstack((g_0_a, g_1_a))

In [6]:
# vettore delle risposte
labels = np.concatenate((np.zeros(N0), np.ones(N1))) # uguale per A e B
original_indices = np.arange(1, 45, step=1)

In [7]:
N_neg = N0
n_pos = N1

### normalizzazione del segnale

In [8]:
# normalizzazione del segnale
from sklearn.preprocessing import MinMaxScaler
matricione = MinMaxScaler().fit_transform(matricione, labels)
matricione_testing = MinMaxScaler().fit_transform(matricione_testing, labels)

### `cross_val_score`

In [9]:
# modulo delle support vector machines 
from sklearn import svm
from sklearn.model_selection import cross_val_score

cv = 5 # default: 5 

- `cross_val_score` valuta lo score di un **estimator** usando il suo metodo interno `.score()`
- `svm.SVC` è l'**estimator** scelto, il suo _score_ è l'**accuracy**: la frazione di pattern correttamente classificati
$$\texttt{accuracy}(y, \hat{y}) = \frac{1}{n_\text{samples}} \sum_{i=0}^{n_\text{samples}-1} 1(\hat{y}_i = y_i)$$
- l'**error** è $1-\texttt{accuracy}(y, \hat{y})$

vogliamo usare la strategia di _cross_validation_ `KFold`:

<p align="center">
  <img src="images/KFold-scheme.png" alt="Schema k-fold cross-validation">
</p>

In [10]:
from sklearn.model_selection import KFold
cv_strategy = KFold(n_splits=6, shuffle=True, random_state=42) 

se viene passato un integer a `cv`, viene usata automaticamente la strategia `KFold`; noi la passiamo esplicitamente.

In [11]:
# bisogna dirgli di essere lineare, oppure usare LinearSVC
estimator = svm.SVC(kernel='linear', random_state=None)
scores = cross_val_score(estimator, matricione, labels, cv=cv_strategy)

`cross_val_score` chiama il metodo `.fit()` di `svm.SVC` sui dati di training, poi il suo metodo `.score()` sui dati di test. 

In [12]:
scores

array([0.75      , 0.75      , 0.71428571, 0.57142857, 0.57142857,
       0.57142857])

In [13]:
media = np.mean(scores)
dev_st = np.std(scores)

print(f"accuracy = {media:.4f} ± {dev_st:.4f}")

accuracy = 0.6548 ± 0.0842


### `cross_val_predict`

In [14]:
from sklearn.model_selection import cross_val_predict
predictions = cross_val_predict(estimator, matricione, labels, cv=cv_strategy)

In [15]:
predictions

array([0., 1., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 1., 1.,
       0., 0., 0., 1., 1., 0., 0., 0., 1., 1., 1., 1., 1., 1., 1., 0., 1.,
       1., 1., 1., 0., 0., 0., 1., 1., 1., 0.])

In [16]:
positivi = predictions[22:]
negativi = predictions[:22]

In [17]:
num_p, num_n = 0, 0
for p, n in zip (positivi, negativi):
    if p == 1:
        num_p += 1
    if n == 0:
        num_n += 1
        
acc = (num_p + num_n)/44
print(acc)

0.6590909090909091


### Manuale

`KFold.split()` non restituisce gli `n_split` array pronti all'uso: **genera** gli indici per un fold ogni volta che viene chiamato, per un totale di `n_split` volte.

> 💡 Provare `Stratified_KFold` per assicurarsi un buon bilanciamento di 0 e 1 in ogni fold. Perché *shuffle* di `KFold` non lo garantisce.

> 💡 Provare a implementare **Principal Component Analysis** (PCA)

In [19]:
from sklearn import svm
from sklearn.model_selection import StratifiedKFold

from sklearn.metrics import accuracy_score # forse ridondante

# scelta dei seed
seed_list = [1_434_512, 432_632, 542_397, 76_301, 97, 7_233, 19_000_234, 500_634, 1, 3242]
kfold_seed = 42

fold_accuracies = {}
mean_accuracies = {}
std_accuracies = {}

best_c = {}
best_kernel = {}
best_model = {}

for seed in seed_list:

    # scelta della strategia di cross-validation
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed) 

    # lista per accuracies
    fold_accuracies[seed] = []
    
    best_score = 0
 
    for i, (train_idx, test_idx) in enumerate( kf.split(matricione, labels) ): 
        
        pattern_train, pattern_test = matricione[train_idx], matricione[test_idx]
        labels_train, labels_test = labels[train_idx], labels[test_idx]
        
                
        my_svm = svm.SVC(kernel='linear', random_state=42)
        
        # fit
        my_svm.fit(pattern_train, labels_train)
        
        # predict
        pred = my_svm.predict(pattern_test)
        
        # score (accuracy)
        #acc = accuracy_score(labels_test, pred)
        sco = my_svm.score(pattern_test, labels_test) # metodo .score di SVC dà l'accuracy di default!
        
        if sco > best_score: 
            best_score = sco
            best_model[seed] = my_svm
                    
        fold_accuracies[seed].append(sco)
        
        #print(f"Fold {i+1}: Accuracy = {acc:.5f} | SVC.score = {sco:.5f}")
    mean_accuracies[seed] = np.mean(fold_accuracies[seed])
    std_accuracies[seed] = np.std(fold_accuracies[seed])

    for s in fold_accuracies[seed]: print(f"{s:.3f}", end=', ')
    print(f"Accuracy media sui fold: {mean_accuracies[seed]:.5f} ± {std_accuracies[seed]:.5f}")
    
    
total_mean = np.mean(list(mean_accuracies.values()))
total_std = np.std(list(mean_accuracies.values()))
print(f"Accuracy media sui seed delle medie sui fold: {total_mean:.3f} ± {total_std:.3f}")

0.667, 0.667, 0.778, 0.778, 0.875, Accuracy media sui fold: 0.75278 ± 0.07876
0.889, 0.889, 0.778, 0.667, 0.750, Accuracy media sui fold: 0.79444 ± 0.08535
0.889, 0.778, 0.667, 0.556, 0.750, Accuracy media sui fold: 0.72778 ± 0.11167
0.778, 0.889, 0.667, 0.667, 0.625, Accuracy media sui fold: 0.72500 ± 0.09639
0.778, 0.778, 0.667, 0.556, 0.750, Accuracy media sui fold: 0.70556 ± 0.08535
0.778, 0.444, 0.667, 0.778, 0.750, Accuracy media sui fold: 0.68333 ± 0.12620
0.778, 0.667, 0.889, 0.667, 0.625, Accuracy media sui fold: 0.72500 ± 0.09639
1.000, 0.889, 0.556, 0.667, 0.625, Accuracy media sui fold: 0.74722 ± 0.16860
0.556, 0.778, 0.889, 0.444, 0.750, Accuracy media sui fold: 0.68333 ± 0.16063
0.667, 0.667, 0.889, 0.556, 0.750, Accuracy media sui fold: 0.70556 ± 0.11055
Accuracy media sui seed delle medie sui fold: 0.725 ± 0.032


Cos'è il segnale? Si può lavorare in un altro spazio? Benchmark

### GridSearch

In [21]:
# parametri da provare
param_grid = [
  {'C': [1E-13, 1E-12, 1E-11, 1E-10], 'kernel': ['linear']},
]

In [22]:
from sklearn import svm
from sklearn.model_selection import StratifiedKFold, GridSearchCV

from sklearn.metrics import accuracy_score # forse ridondante

# scelta dei seed
seed_list = [1_434_512, 432_632, 542_397, 76_301, 97, 7_233, 19_000_234, 500_634, 1, 3242]
kfold_seed = 42

best_c = {}
best_accuracies = {}

for seed in seed_list:
    print(f"🚀 SEED: {seed}")
    # istanza modello ⚠️ random_state qui non fa niente
    my_svm = svm.SVC()#kernel='linear', random_state=42)
    # scelta della strategia di cross-validation
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed) 
    
    # gridsearch: proverà diverse combinazioni di parametri
    gs = GridSearchCV(my_svm, cv=kf, param_grid=param_grid)
    
    # fit
    gs.fit(matricione, labels)
    
    # controllo parametri
    print(" - Migliori iperparametri:", gs.best_params_)
    print(" - Miglior score:", gs.best_score_)

    best_c[seed] = gs.best_params_['C']
    best_accuracies[seed] = gs.best_score_
    
media_accuracies = np.mean(list(best_accuracies.values()))
std_accuracies = np.std(list(best_accuracies.values()))
print(f"MEDIA: {media_accuracies} ± {std_accuracies}")

🚀 SEED: 1434512
 - Migliori iperparametri: {'C': 1e-13, 'kernel': 'linear'}
 - Miglior score: 0.5305555555555556
🚀 SEED: 432632
 - Migliori iperparametri: {'C': 1e-13, 'kernel': 'linear'}
 - Miglior score: 0.4805555555555555
🚀 SEED: 542397
 - Migliori iperparametri: {'C': 1e-13, 'kernel': 'linear'}
 - Miglior score: 0.4805555555555555
🚀 SEED: 76301
 - Migliori iperparametri: {'C': 1e-13, 'kernel': 'linear'}
 - Miglior score: 0.45555555555555555
🚀 SEED: 97
 - Migliori iperparametri: {'C': 1e-13, 'kernel': 'linear'}
 - Miglior score: 0.4805555555555555
🚀 SEED: 7233
 - Migliori iperparametri: {'C': 1e-13, 'kernel': 'linear'}
 - Miglior score: 0.4805555555555555
🚀 SEED: 19000234
 - Migliori iperparametri: {'C': 1e-13, 'kernel': 'linear'}
 - Miglior score: 0.45555555555555555
🚀 SEED: 500634
 - Migliori iperparametri: {'C': 1e-13, 'kernel': 'linear'}
 - Miglior score: 0.45555555555555555
🚀 SEED: 1
 - Migliori iperparametri: {'C': 1e-13, 'kernel': 'linear'}
 - Miglior score: 0.530555555555555

Si potrebbe affermare che C = 1e-05 è quello selezionato la maggior parte delle volte (moda di C)

## PCA

La PCA si fa solo sui dati usati per il training
- o usi tutto segnale B per training e quindi fai PCA su segnale A
- o usi 80% segnale B per training e 20% per test interno, quindi PCA lo fai solo su 20% di B

In [25]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 1. Prepara i dati (la PCA va SEMPRE fatta dopo lo scaling)
scaler = StandardScaler()
data_scaled = scaler.fit_transform(matricione)

# 2. Fit della PCA su TUTTE le componenti possibili
pca = PCA() 
pca.fit(data_scaled)

# 3. Ottenere i "pesi relativi" (Autovalori normalizzati)
autovalori_relativi = pca.explained_variance_ratio_

# 4. Calcolare la Varianza Cumulata
varianza_cumulata = np.cumsum(autovalori_relativi)

print(f"Varianza spiegata dalle prime N componenti:")
for N in range(10):
    print(f"{varianza_cumulata[N]*100:.2f}%")

Varianza spiegata dalle prime N componenti:
16.18%
25.39%
32.27%
37.86%
42.63%
47.07%
50.76%
54.37%
57.75%
60.93%
